# Elecciones presidenciales de EE. UU. (1976–2020)

**Análisis exploratorio reproducible**  
Fuentes: Federal Election Commission (resultados presidenciales) y U.S. House Office of the Historian (control histórico). Véanse la fundamentación y la trazabilidad en `../docs/fundamentacion.md` y `../data/README.md`.

**Pregunta:** ¿qué relación existe entre el margen del voto popular, el margen electoral, el voto a otras candidaturas y el volumen de votos? Las asociaciones que siguen no implican causalidad.


## 1. Carga y validación

Se usan solo módulos de la biblioteca estándar. La ruta funciona al ejecutar desde la raíz del repositorio o desde `notebooks/`.

In [ ]:
from csv import DictReader
from pathlib import Path
from math import sqrt
from statistics import mean, median

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "elecciones_presidenciales_1976_2020.csv"
OUTPUT = ROOT / "notebooks" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)

with DATA.open(encoding="utf-8", newline="") as source:
    rows = list(DictReader(source))

integer_columns = [
    "year", "dem_votes", "rep_votes", "other_votes", "total_votes",
    "dem_electoral", "rep_electoral", "other_electoral",
]
for row in rows:
    for column in integer_columns:
        row[column] = int(row[column])

print(f"Observaciones: {len(rows)}; período: {rows[0]['year']}–{rows[-1]['year']}")
print("Columnas:", ", ".join(rows[0]))


In [ ]:
years = [row["year"] for row in rows]
assert years == list(range(1976, 2021, 4))
assert len(years) == len(set(years))
for row in rows:
    assert row["dem_votes"] + row["rep_votes"] + row["other_votes"] == row["total_votes"]
    assert row["dem_electoral"] + row["rep_electoral"] + row["other_electoral"] == 538
print("Validaciones superadas: periodicidad, unicidad y sumas popular/electoral.")


## 2. Construcción de indicadores

El margen popular se calcula sobre el voto **bipartidista**; el margen electoral, sobre los 538 electores. Ambos son positivos si favorecen al Partido Demócrata. `other_share` conserva el peso de las demás candidaturas sobre el total.

In [ ]:
for row in rows:
    two_party = row["dem_votes"] + row["rep_votes"]
    row["dem_two_party_pct"] = 100 * row["dem_votes"] / two_party
    row["popular_margin_pp"] = 100 * (row["dem_votes"] - row["rep_votes"]) / two_party
    row["electoral_margin_pp"] = 100 * (row["dem_electoral"] - row["rep_electoral"]) / 538
    row["other_share_pct"] = 100 * row["other_votes"] / row["total_votes"]
    row["popular_winner"] = "Democratic" if row["popular_margin_pp"] > 0 else "Republican"
    row["mismatch"] = row["popular_winner"] != row["winner"]

print("Año | margen popular (pp) | margen electoral (pp) | otros (%) | discrepancia")
for row in rows:
    print(f"{row['year']} | {row['popular_margin_pp']:>7.2f} | {row['electoral_margin_pp']:>7.2f} | "
          f"{row['other_share_pct']:>5.2f} | {'sí' if row['mismatch'] else 'no'}")


## 3. Estadística descriptiva

In [ ]:
def describe(values):
    return {
        "n": len(values), "mín": min(values), "media": mean(values),
        "mediana": median(values), "máx": max(values),
    }

for variable in ("popular_margin_pp", "electoral_margin_pp", "other_share_pct", "total_votes"):
    summary = describe([row[variable] for row in rows])
    print(variable, {key: round(value, 2) if isinstance(value, float) else value
                     for key, value in summary.items()})


## 4. Correlaciones

Pearson resume asociación lineal; Spearman calcula Pearson sobre rangos y resume asociación monotónica. Con solo 12 elecciones y dependencia temporal, los coeficientes son descriptivos: no se realizan pruebas de significación.

In [ ]:
def pearson(xs, ys):
    xbar, ybar = mean(xs), mean(ys)
    numerator = sum((x - xbar) * (y - ybar) for x, y in zip(xs, ys))
    denominator = sqrt(sum((x - xbar) ** 2 for x in xs) * sum((y - ybar) ** 2 for y in ys))
    return numerator / denominator

def ranks(values):
    # Rango promedio en empates.
    result = [0.0] * len(values)
    ordered = sorted(enumerate(values), key=lambda pair: pair[1])
    start = 0
    while start < len(ordered):
        end = start
        while end + 1 < len(ordered) and ordered[end + 1][1] == ordered[start][1]:
            end += 1
        average_rank = (start + end + 2) / 2
        for position in range(start, end + 1):
            result[ordered[position][0]] = average_rank
        start = end + 1
    return result

def spearman(xs, ys):
    return pearson(ranks(xs), ranks(ys))

variables = ["popular_margin_pp", "electoral_margin_pp", "other_share_pct", "total_votes"]
print("Par | Pearson | Spearman")
for index, left in enumerate(variables):
    for right in variables[index + 1:]:
        xs, ys = [r[left] for r in rows], [r[right] for r in rows]
        print(f"{left} × {right} | {pearson(xs, ys): .3f} | {spearman(xs, ys): .3f}")


## 5. Visualización reproducible

El código produce SVG sin bibliotecas externas. Las etiquetas permiten inspeccionar qué elección origina cada punto.

In [ ]:
def scatter_svg(data, x_key, y_key, x_label, y_label, path):
    width, height, pad = 760, 470, 70
    xs, ys = [r[x_key] for r in data], [r[y_key] for r in data]
    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
    xgap, ygap = (xmax - xmin or 1) * .1, (ymax - ymin or 1) * .1
    xmin, xmax, ymin, ymax = xmin-xgap, xmax+xgap, ymin-ygap, ymax+ygap
    sx = lambda x: pad + (x-xmin)/(xmax-xmin)*(width-2*pad)
    sy = lambda y: height-pad - (y-ymin)/(ymax-ymin)*(height-2*pad)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="white"/>',
             f'<line x1="{pad}" y1="{height-pad}" x2="{width-pad}" y2="{height-pad}" stroke="#333"/>',
             f'<line x1="{pad}" y1="{pad}" x2="{pad}" y2="{height-pad}" stroke="#333"/>']
    if xmin <= 0 <= xmax:
        parts.append(f'<line x1="{sx(0):.1f}" y1="{pad}" x2="{sx(0):.1f}" y2="{height-pad}" stroke="#bbb" stroke-dasharray="4"/>')
    if ymin <= 0 <= ymax:
        parts.append(f'<line x1="{pad}" y1="{sy(0):.1f}" x2="{width-pad}" y2="{sy(0):.1f}" stroke="#bbb" stroke-dasharray="4"/>')
    for row in data:
        color = "#2563eb" if row["winner"] == "Democratic" else "#dc2626"
        x, y = sx(row[x_key]), sy(row[y_key])
        parts += [f'<circle cx="{x:.1f}" cy="{y:.1f}" r="6" fill="{color}" opacity=".8"/>',
                  f'<text x="{x+8:.1f}" y="{y-7:.1f}" font-size="11" font-family="sans-serif">{row["year"]}</text>']
    parts += [f'<text x="{width/2}" y="{height-18}" text-anchor="middle" font-size="14" font-family="sans-serif">{x_label}</text>',
              f'<text transform="translate(18 {height/2}) rotate(-90)" text-anchor="middle" font-size="14" font-family="sans-serif">{y_label}</text>',
              '<text x="70" y="25" font-size="12" font-family="sans-serif" fill="#2563eb">● ganador demócrata</text>',
              '<text x="230" y="25" font-size="12" font-family="sans-serif" fill="#dc2626">● ganador republicano</text>', '</svg>']
    path.write_text("\n".join(parts), encoding="utf-8")
    return path

plot = scatter_svg(rows, "popular_margin_pp", "electoral_margin_pp",
                   "Margen popular bipartidista (pp)", "Margen electoral (pp)",
                   OUTPUT / "margen_popular_vs_electoral.svg")
print("Gráfico guardado en:", plot.relative_to(ROOT))


![Margen popular frente al electoral](output/margen_popular_vs_electoral.svg)

## 6. Casos discordantes e interpretación

In [ ]:
discordant = [row for row in rows if row["mismatch"]]
print("Elecciones con ganador popular bipartidista distinto del ganador electoral:")
for row in discordant:
    print(f"- {row['year']}: voto popular → {row['popular_winner']}; Colegio Electoral → {row['winner']}")

r = pearson([row["popular_margin_pp"] for row in rows],
            [row["electoral_margin_pp"] for row in rows])
print(f"\nCorrelación lineal entre márgenes: {r:.3f} (n={len(rows)}).")


### Lectura

Se espera una asociación positiva entre ambos márgenes, pero no una equivalencia: el Colegio Electoral agrega resultados estatales y puede amplificar un margen o invertir al ganador nacional. Los casos discordantes deben analizarse, no eliminarse como errores. El porcentaje de terceras fuerzas tampoco identifica transferencia de votos entre candidaturas.

### Conclusión y próximos pasos

Este EDA establece una base auditada para formular hipótesis, no una explicación causal. Un análisis posterior debería usar estado–año como unidad, sumar población elegible y variables sociodemográficas de Census/ACS, y modelar tendencia temporal y heterogeneidad estatal. La cantidad pequeña de observaciones y el carácter agregado impiden extrapolar estas correlaciones a votantes individuales (falacia ecológica).
